In [41]:
# Libraries
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models

from sentence_transformers import SentenceTransformer
import uuid, textwrap


In [25]:
MODEL_NAME = "llama3.1"
provider = OllamaProvider(base_url="http://localhost:11434/v1")
llm = OpenAIChatModel(provider=provider, model_name=MODEL_NAME)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM = embedder.get_sentence_embedding_dimension()

COLLECTION = "notebook_demo"

qdrant = QdrantClient(":memory:")
qdrant.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=qdrant_models.VectorParams(size=EMBED_DIM, distance=qdrant_models.Distance.COSINE),
)

/var/folders/yg/5pzxthwx17n44jwy6635l7mw0000gn/T/ipykernel_7786/2383357056.py:11: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

In [26]:
agent = Agent(model=llm)
result = await agent.run("Hello, world!")
print(result.output)

Hello there, human from planet Earth! It's nice to meet you. Is there something I can help you with, or would you like to chat about the latest in artificial intelligence, space exploration, or perhaps the best pizza toppings?


In [27]:
def chunk_text(s: str, max_chars=800):
    s = " ".join(s.split())
    for i in range(0, len(s), max_chars):
        yield s[i:i+max_chars]

def upsert_texts(texts, source="manual"):
    points = []
    for t in texts:
        for ch in chunk_text(t):
            vec = embedder.encode(ch).tolist()
            points.append(
                qdrant_models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vec,
                    payload={"text": ch, "source": source},
                )
            )
    if points:
        qdrant.upsert(collection_name=COLLECTION, points=points)

def retrieve(query, k=5):
    qvec = embedder.encode(query).tolist()
    hits = qdrant.search(collection_name=COLLECTION, query_vector=qvec, limit=k)
    return [
        {
            "text": h.payload["text"],
            "source": h.payload.get("source", "unknown"),
            "score": float(h.score),
        }
        for h in hits
    ]


In [28]:
doc1 = """
Universal Basic Income (UBI) is a periodic cash payment unconditionally delivered to all on an individual basis,
without means-test or work requirement. Advocates claim it reduces poverty and simplifies welfare administration.
Critics argue it may disincentivize work and be fiscally challenging at scale.
"""

doc2 = """
Evidence from small-scale pilots (e.g., Finland, Namibia, certain US city pilots) suggests modest improvements in
financial security and well-being. Impacts on labor supply are mixed and often small in short-term trials.
Funding designs (e.g., taxes, dividend models) strongly affect feasibility.
"""

upsert_texts([doc1, doc2], source="demo-notes")


In [29]:
def build_context(query, k=5):
    hits = retrieve(query, k=k)
    context = "\n\n".join(
        f"[{i+1}] {h['text']}\n— source: {h['source']} (score={h['score']:.3f})"
        for i, h in enumerate(hits)
    )
    return context, hits

research_agent = Agent(
    llm,
    system_prompt="""
You are a careful analyst. Use the provided CONTEXT to answer the QUESTION.
Cite passages using [#] from the context when relevant. If context is thin, say what’s missing.
""".strip(),
)

question = "Is universal basic income a good idea? Summarize pros and cons."
context, hits = build_context(question, k=5)

prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nAnswer:"
result = await research_agent.run(prompt)
print(result.output)


/var/folders/yg/5pzxthwx17n44jwy6635l7mw0000gn/T/ipykernel_7786/162813535.py:23: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(collection_name=COLLECTION, query_vector=qvec, limit=k)


Based on the provided context, here is a summary of pros and cons of Universal Basic Income:

**Pros:**

* Reduces poverty (#1)
* Simplifies welfare administration (#1)
* Provides modest improvements in financial security and well-being [#2]

**Cons:**

* May disincentivize work (#1)
* Can be fiscally challenging at scale (#1)
* Impacts on labor supply are mixed and often small in short-term trials[#2]
* Funding designs can affect feasibility, with some options potentially being more feasible than others [#2]


In [30]:
hits


[{'text': 'Universal Basic Income (UBI) is a periodic cash payment unconditionally delivered to all on an individual basis, without means-test or work requirement. Advocates claim it reduces poverty and simplifies welfare administration. Critics argue it may disincentivize work and be fiscally challenging at scale.',
  'source': 'demo-notes',
  'score': 0.6261132027022887},
 {'text': 'Evidence from small-scale pilots (e.g., Finland, Namibia, certain US city pilots) suggests modest improvements in financial security and well-being. Impacts on labor supply are mixed and often small in short-term trials. Funding designs (e.g., taxes, dividend models) strongly affect feasibility.',
  'source': 'demo-notes',
  'score': 0.39588954176804797}]

In [39]:
from pydantic_ai import RunContext

@agent.tool
def ddg_search_web(ctx: RunContext[None], query: str, max_results: int = 5) -> list[str]:
    """
    Quick DuckDuckGo HTML search. Returns a list of 'title — url' strings.
    """
    import requests
    from bs4 import BeautifulSoup

    r = requests.get("https://duckduckgo.com/html/", params={"q": query}, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    out = []
    for a in soup.select(".result__a")[:max_results]:
        title = a.get_text(" ", strip=True)
        url = a.get("href")
        out.append(f"{title} — {url}")
    return out

@agent.tool
def retrieve_ctx(ctx: RunContext[None], query: str, k: int = 5) -> str:
    hits = retrieve(query, k=k)
    lines = []
    for i, h in enumerate(hits, start=1):
        lines.append(f"[{i}] {h['text']}\n— source: {h['source']} (score={h['score']:.3f})")
    return "\n\n".join(lines)



In [40]:
gent = Agent(
    model=llm,
    system_prompt=(
        "You may call tools. If you need recent or missing context, first call `ddg_search` "
        "to collect links, then call `retrieve_ctx` with the question to ground your answer. "
        "Answer with citations like [#] if you used retrieved context."
    ),
    tools=[retrieve_ctx, ddg_search_web],
)

res = await agent.run("Give me current arguments for/against UBI and cite retrieved context.")
print(res.output)

Based on the tool call response, here are some current arguments for and against Universal Basic Income (UBI):

**Arguments For UBI:**

1. **Poverty Reduction**: Providing every citizen with a basic income can help reduce poverty and income inequality. (Source: "A Basic Income Guarantee Can Solve Poverty" by The Guardian)
2. **Simplified Welfare System**: UBI simplifies the welfare system, reducing bureaucracy and costs associated with administering multiple benefits. (Source: "The Case for Universal Basic Income" by The Economist)
3. **Freedom to Choose Work**: UBI allows individuals to choose work that is fulfilling, rather than just taking a job for financial reasons. (Source: "Why We Need a Universal Basic Income" by Forbes)

**Arguments Against UBI:**

1. **Cost and Funding**: Implementing UBI would require significant funding, which could be difficult to finance without increasing taxes or cutting other social programs. (Source: "The Costs of Universal Basic Income" by The Herita

In [1]:
# SMOKE test

In [3]:
# In a notebook/REPL
from agentic_research.rag.qdrant_store import QdrantRAG
from agentic_research.tools.web_tools import make_tools

rag = QdrantRAG()
tools = make_tools(rag)

# Try web search
hits = tools[1].function("universal basic income pros and cons", max_results=3)  # web_search_tool
print(hits)

# # Try ingest + retrieve
# tools[3].function("universal basic income pros and cons", max_results=3)  # web_search_and_ingest
# print(rag.query("UBI effects", k=3))


/Users/harnoorrangi/Documents/Coding/Projects/agentic-research/src/agentic_research/rag/ingest.py:10: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[{'title': 'Universal GSM Software [Archive] - GSM-Forum', 'url': 'https://forum.gsmhosting.com/vbb/archive/f-162.html', 'snippet': '[Archive] Write here for questions about Universal GSM Software. Example: USPU, twinsim, sms-soft, drivers...'}, {'title': 'UnlockTool-2024.08.15.0 Released Update - GSM-Forum', 'url': 'https://forum.gsmhosting.com/vbb/f1112/unlocktool-2024-08-15-0-released-update-3328651/', 'snippet': "UnlockTool 2024.08.15.0 Released Update Change Logs : Bug Fix MTK Chip V6 Erase FRP in Tab Model & MTK Universal TECNO/INFINIX - We're added the"}, {'title': 'Sigma Plus Software v1.00.03.03 - Universal Loaders - GSM-Forum', 'url': 'https://forum.gsmhosting.com/vbb/f719/sigma-plus-software-v1-00-03-03-universal-loaders-3240655/', 'snippet': 'Dec 21, 2023 · Sigma Plus Software v1.00.03 Unisoc Tab: 1. Released solution that allows to work with various UNISOC-models on the market. How to use: If you know'}]


In [8]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent
from agentic_research.config.config import settings

llm = OpenAIChatModel(
    model_name=settings.model_name,
    provider=OllamaProvider(base_url=settings.openai_base_url),
)

agent = Agent(model=llm, tools=tools, system_prompt="Use tools if necessary and cite sources.")
res = await agent.run("Should I invest in RESP (Registered Education Savings Plan) or TFSA (Tax-Free Savings Account) for my child's education savings?")
print(res.output)


/Users/harnoorrangi/Documents/Coding/Projects/agentic-research/src/agentic_research/rag/ingest.py:10: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Based on the search results, it seems that RESP is a more popular choice for education savings. Here are some key points to consider:

* RESP allows you to earn investment income tax-free and withdraw funds tax-free if used for qualified education expenses.
* The government also contributes up to 20% (40% in Quebec) of your contributions annually, up to a maximum of $7,200 per year.
* However, the Canada Education Savings Grant (CESG) contribution is not guaranteed and may vary depending on your income.

On the other hand, TFSAs offer more flexibility as the contributions do not have limits. The interest earned on TFSA will be tax-free once you start withdrawing.

It's generally recommended to use a combination of both RESP and TFSA for education savings, depending on your family's financial situation and goals. It's always best to consult with a financial advisor before making any investment decisions.

If you're considering combining RESP and TFSA for education savings, you could con

In [1]:
from agentic_research.rag.ingest import ddg_reddit_search

2025-10-05 20:22:14.022 | DEBUG    | agentic_research.config.config:<module>:49 - Settings loaded: model=llama3.1 qdrant_url=None
/Users/harnoorrangi/Documents/Coding/Projects/agentic-research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ddg_reddit_search("universal basic income", max_results=3)

2025-10-05 20:22:36.729 | INFO     | agentic_research.rag.ingest:ddg_reddit_search:17 - DDG reddit search: query='universal basic income' max_results=3
2025-10-05 20:22:36.730 | INFO     | agentic_research.rag.ingest:ddg_search:12 - DDG search: query='site:reddit.com universal basic income' max_results=3


[{'title': 'Is universal basic income possible here? : BasicIncome',
  'href': 'https://www.reddit.com/r/BasicIncome/comments/14u4w26/is_universal_basic_income_possible_here/',
  'body': 'Basic Income is alternately referred to as a guaranteed annual income , citizen s ... Is universal basic income possible here? ( newstatesman.com )'},
 {'title': 'Universal basic income: Plans drawn up for £1,600 a month',
  'href': 'https://www.reddit.com/r/BasicIncome/comments/142hdpe/universal_basic_income_plans_drawn_up_for_1600_a/',
  'body': '... million question: Could Universal Basic Income ... Universal basic income : Plans drawn up for £1,600 a month trial in England - BBC News ( bbc.com )'},
 {'title': 'Universal Basic Income is affordable and feasible: evidence',
  'href': 'https://www.reddit.com/r/BasicIncome/comments/14n1726/universal_basic_income_is_affordable_and_feasible/',
  'body': 'Universal Basic Income is affordable and feasible: evidence from UK economic microsimulation modellin